# 01 – Tensor-Basics: PyTorch von Grund auf

**Lernziele:**
- Tensoren erstellen, inspizieren und manipulieren
- Grundoperationen (elementweise, Matrix-Multiplikation, Reduktion)
- GPU-Unterstützung mit CUDA
- Autograd: automatische Differentiation verstehen
- Erste Trainingsschleife: Lineare Regression von Hand

---

In [ ]:
import numpy as np
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar:  {torch.cuda.is_available()}")

## 1. Tensoren erstellen

Ein **Tensor** ist die fundamentale Datenstruktur in PyTorch – vergleichbar mit NumPy-Arrays, aber mit GPU-Support und Autograd.

In [ ]:
# Aus Liste
t = torch.tensor([1, 2, 3, 4])
print(f"torch.tensor([1,2,3,4])     → {t}")

# Nullen, Einsen, Zufall
print(f"torch.zeros(2,3)            → \n{torch.zeros(2,3)}")
print(f"torch.ones(2,3)             → \n{torch.ones(2,3)}")
print(f"torch.randn(2,3)            → \n{torch.randn(2,3)}")

# Aus NumPy
np_arr = np.array([1.0, 2.0, 3.0])
t_np = torch.from_numpy(np_arr)
print(f"torch.from_numpy(np.array)  → {t_np}")

## 2. Tensor-Attribute

Jeder Tensor hat wichtige Metadaten: Shape, Datentyp, Gerät (CPU/GPU).

In [ ]:
x = torch.randn(2, 3, 4)
print(f"shape:    {x.shape}")
print(f"dtype:    {x.dtype}")
print(f"device:   {x.device}")
print(f"ndim:     {x.ndim}")

## 3. Grundoperationen

**Wichtig:** `*` ist elementweise Multiplikation, `@` ist Matrix-Multiplikation (Skalarprodukt).

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a + b     = {a + b}")
print(f"a * b     = {a * b}        (elementweise!)")
print(f"a @ b     = {a @ b:.1f}    (Skalarprodukt)")
print(f"a.mean()  = {a.mean():.1f}")
print(f"a.sum()   = {a.sum():.1f}")

## 4. Reshape & Indexing

`reshape` und `view` ändern die Form eines Tensors. `view` teilt den Speicher, `reshape` kopiert bei Bedarf.

In [ ]:
x = torch.arange(12)
print(f"arange(12)           → {x}")
print(f".reshape(3,4)        → \n{x.reshape(3,4)}")
print(f".view(4,3)           → \n{x.view(4,3)}")
print(f"x[2:5]               → {x[2:5]}")

## 5. GPU-Verfügbarkeit

PyTorch macht GPU-Computing einfach: `.cuda()` oder `.to(device)` verschiebt Tensoren auf die GPU.

In [ ]:
print(f"CUDA verfügbar:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    gpu_t = torch.randn(1000, 1000).cuda()
    print(f"Tensor auf GPU:  {gpu_t.device}")
else:
    print("Keine GPU verfügbar – Training läuft auf CPU.")

---
## 6. Autograd – Automatische Gradienten

`requires_grad=True` aktiviert die automatische Differentiation. `backward()` berechnet die Gradienten.

### Einfaches Beispiel: $f(x) = x^2$ bei $x=3$

Die Ableitung ist $f'(x) = 2x$, also $f'(3) = 6$.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print(f"x = {x.item()}, y = x² = {y.item()}")
print(f"dy/dx = 2x = {x.grad.item()}  (erwartet: 6.0) ✓")

### Kettenregel: $f(x) = (2x + 1)^2$

Ableitung: $f'(x) = 2 \cdot (2x+1) \cdot 2 = 4(2x+1)$. Bei $x=2$: $f'(2) = 20$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = (2 * x + 1) ** 2
y.backward()
print(f"x = {x.item()}, y = (2x+1)² = {y.item()}")
print(f"dy/dx = 4(2x+1) = {x.grad.item():.1f}  (erwartet: 20.0) ✓")

### Mehrere Variablen: $z = x^2 + y^3$

Partielle Ableitungen: $\frac{\partial z}{\partial x} = 2x$, $\frac{\partial z}{\partial y} = 3y^2$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x**2 + y**3
z.backward()
print(f"x={x.item()}, y={y.item()}, z = {z.item()}")
print(f"∂z/∂x = 2x = {x.grad.item():.1f}  (erwartet: 4.0) ✓")
print(f"∂z/∂y = 3y² = {y.grad.item():.1f}  (erwartet: 27.0) ✓")

### ⚠️ Achtung: Gradienten akkumulieren!

PyTorch **akkumuliert** Gradienten bei jedem `backward()`-Aufruf. Deshalb muss man `optimizer.zero_grad()` aufrufen – sonst addieren sich die Gradienten über mehrere Batches.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    loss = w ** 2
    loss.backward()
    print(f"Schritt {i+1}: w.grad = {w.grad.item():.1f}  "
          f"(akkumuliert: {2*w.item()*(i+1):.1f})")

---
## 7. Lineare Regression mit PyTorch – Erstes Training

Wir trainieren ein einfaches lineares Modell $y = wx + b$ auf synthetischen Daten. **Ohne** `nn.Module` oder `Optimizer` – alles von Hand, um die Mechanik zu verstehen.

In [ ]:
# Daten generieren: y = 3x + 2 + Rauschen
torch.manual_seed(42)
N = 100
X = torch.randn(N, 1) * 2
y = 3 * X + 2 + torch.randn(N, 1) * 0.5

print(f"Daten: {N} Punkte, y = 3x + 2 + noise")

# Parameter initialisieren
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.01
epochs = 100

print(f"\nTraining ({epochs} Epochen, lr={lr}):")
for epoch in range(epochs):
    # Forward
    y_pred = X @ w + b
    loss = ((y_pred - y) ** 2).mean()

    # Backward
    loss.backward()

    # Update (manuell, ohne Optimizer)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

    if epoch % 20 == 0:
        print(f"  Epoche {epoch:3d}: loss={loss.item():.4f}, "
              f"w={w.item():.3f}, b={b.item():.3f}")

print(f"\n✅ Gefunden: w={w.item():.3f}, b={b.item():.3f}")
print("   Erwartet:  w=3.0, b=2.0")

---
## Zusammenfassung

| Konzept | Beschreibung |
|---|---|
| **Tensor** | N-dimensionale Datenstruktur (wie NumPy + GPU + Autograd) |
| **`requires_grad`** | Aktiviert Gradienten-Tracking |
| **`backward()`** | Berechnet Gradienten via Backpropagation |
| **`torch.no_grad()`** | Deaktiviert Gradienten-Tracking (für Inference/Updates) |
| **`zero_grad()`** | Setzt Gradienten zurück (Akkumulation vermeiden!) |
| **`.cuda()` / `.to(device)`** | Verschiebt Tensor auf GPU |

**Nächstes Notebook:** `02_neural_network.ipynb` – Neuronale Netze mit `nn.Module`, DataLoader und Optimizer.